Static cropping

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

RAW_VIDEO = '/mnt/Data/Projects/cloud_deployment/videos/Trial3/20260121_SERt3tp_final_sync.mp4'

# Extract the first frame
cap = cv2.VideoCapture(RAW_VIDEO)
ret, frame = cap.read()
cap.release()

if ret:
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(30, 16), dpi=300)
    ax.imshow(frame)
    
    # Draw 200px grid
    ax.set_xticks(range(0, frame.shape[1], 200))
    ax.set_yticks(range(0, frame.shape[0], 200))
    ax.grid(color='red', linestyle='-', linewidth=1, alpha=0.7)
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(fontsize=8)
    
    plt.savefig('trial3_grid_reference.png', bbox_inches='tight', pad_inches=0.1)
    print("✅ Grid saved as 'trial3_grid_reference.png'. Open it to find your crop coordinates.")
else:
    print("❌ Failed to read video.")

Once you check the grid image, run this cell using the ! bash command. Replace the W:H:X:Y variables with your actual crop dimensions (e.g., crop=2000:2160:900:0 if you want a 2000px wide video starting at X=900).

%%bash
# Replace W:H:X:Y with your grid coordinates
ffmpeg -y -i /mnt/Data/Projects/cloud_deployment/videos/Trial3/20260121_SERt3tp_final_sync.mp4 \
-filter:v "crop=W:H:X:Y" \
-c:a copy \
/mnt/Data/Projects/cloud_deployment/videos/Trial3/20260121_SERt3tp_static_crop.mp4

Cell 3: Stage 1 Setup (The Tracker)
Initializes the project to track only TailBase.

In [ ]:
import deeplabcut
import ruamel.yaml
import os

WORKING_DIR = '/mnt/Data/Projects/cloud_deployment/Scripts'
STATIC_VIDEO = '/mnt/Data/Projects/cloud_deployment/videos/Trial3/20260121_SERt3tp_static_crop.mp4'

print("⚙️ Initializing Stage 1 Project...")
stage1_config = deeplabcut.create_new_project(
    'Stage1_Tracker', 'Dev', [STATIC_VIDEO], 
    working_directory=WORKING_DIR, copy_videos=False
)

# Edit config for single point tracking and MobileNet
ryaml = ruamel.yaml.YAML()
ryaml.preserve_quotes = True
with open(stage1_config, 'r') as f:
    cfg = ryaml.load(f)

cfg['bodyparts'] = ['TailBase']
cfg['skeleton'] = [] # No skeleton needed for a single point
cfg['default_net_type'] = 'mobilenet_v2_1.0'
cfg['numframes2pick'] = 100

with open(stage1_config, 'w') as f:
    ryaml.dump(cfg, f)

print(f"✅ Stage 1 initialized. Extracting frames...")
deeplabcut.extract_frames(stage1_config, mode='automatic', algo='kmeans', userfeedback=False)

Cell 4: Stage 1 Parse & Train
(Assumption: You have labeled the Stage 1 frames in Label Studio and downloaded the CSV). Update the LS_CSV path, then run this block to train the tracker and analyze the static video.

In [ ]:
import pandas as pd
import json
import numpy as np
import deeplabcut
import glob
import re
import yaml

LS_CSV = '/path/to/your/stage1_labelstudio_export.csv'
VIDEO_NAME = '20260121_SERt3tp_static_crop'
PROJECT_PATH = os.path.dirname(stage1_config)

# --- 1. Parser ---
df_ls = pd.read_csv(LS_CSV)
data_rows = []
for _, row in df_ls.iterrows():
    fname = os.path.basename(row['image']).split('-')[-1] if '-' in row['image'] else os.path.basename(row['image'])
    frame_data = {'image': f"labeled-data/{VIDEO_NAME}/{fname}", 'TailBase_x': np.nan, 'TailBase_y': np.nan}
    try:
        labels = json.loads(row['label' if 'label' in df_ls.columns else 'annotation'])
        if isinstance(labels, list) and 'result' in labels[0]: labels = labels[0]['result']
        for l in labels:
            v = l.get('value', {})
            if v.get('keypointlabels', [None])[0] == 'TailBase':
                frame_data['TailBase_x'] = (v['x'] * v.get('original_width', 3840)) / 100.0
                frame_data['TailBase_y'] = (v['y'] * v.get('original_height', 2160)) / 100.0
    except: pass
    data_rows.append(frame_data)

df_final = pd.DataFrame(data_rows).set_index('image')
df_final.columns = pd.MultiIndex.from_product([['Dev'], ['TailBase'], ['x', 'y']], names=['scorer', 'bodyparts', 'coords'])
frame_dest = os.path.join(PROJECT_PATH, 'labeled-data', VIDEO_NAME)
df_final.to_hdf(os.path.join(frame_dest, 'CollectedData_Dev.h5'), key='df_with_missing', mode='w')
df_final.to_csv(os.path.join(frame_dest, 'CollectedData_Dev.csv'))

# --- 2. Dataset & Posix Patch ---
def posix_path_constructor(loader, node):
    if isinstance(node, yaml.ScalarNode): return str(loader.construct_scalar(node))
    elif isinstance(node, yaml.SequenceNode): return str(loader.construct_sequence(node)[0])
    return str(node.value)
yaml.SafeLoader.add_constructor('tag:yaml.org,2002:python/object/apply:pathlib.PosixPath', posix_path_constructor)

deeplabcut.create_training_dataset(stage1_config, net_type='mobilenet_v2_1.0', augmenter_type='default')

# --- 3. VRAM Limits ---
for p in glob.glob(os.path.join(PROJECT_PATH, 'dlc-models', '*', '*', '*', 'pose_cfg.yaml'), recursive=True):
    with open(p, 'r') as f: text = f.read()
    text = re.sub(r'^global_scale:\s*.*$', 'global_scale: 0.4', text, flags=re.MULTILINE)
    if 'train' in p: text = re.sub(r'^batch_size:\s*.*$', 'batch_size: 2', text, flags=re.MULTILINE)
    with open(p, 'w') as f: f.write(text)

# --- 4. Train & Analyze ---
deeplabcut.train_network(stage1_config, maxiters=50000, displayiters=1000, allow_growth=True)
deeplabcut.analyze_videos(stage1_config, [STATIC_VIDEO], save_as_csv=True, batchsize=2)

Cell 5: The Dynamic Cropping Engine
This reads the output from Stage 1. It cuts an 800x800 box centered on TailBase for every frame and writes the _dynamic_crop.mp4. It also saves crop_offsets.csv so we can map the coordinates back later.

In [ ]:
import cv2
import pandas as pd
import numpy as np

# Find the Stage 1 analysis CSV
csv_files = glob.glob(os.path.join(os.path.dirname(STATIC_VIDEO), f'{VIDEO_NAME}DLC*.csv'))
track_csv = sorted(csv_files)[-1] # Gets the latest one

df = pd.read_csv(track_csv, header=[1, 2], index_col=0)
x_coords = df[('TailBase', 'x')].values
y_coords = df[('TailBase', 'y')].values

CROP_SIZE = 800
offsets = []

cap = cv2.VideoCapture(STATIC_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_path = '/mnt/Data/Projects/cloud_deployment/videos/Trial3/20260121_SERt3tp_dynamic_crop.mp4'
out = cv2.VideoWriter(out_path, fourcc, fps, (CROP_SIZE, CROP_SIZE))

frame_idx = 0
print("✂️ Generating dynamically cropped video...")
while True:
    ret, frame = cap.read()
    if not ret: break
    
    h, w, _ = frame.shape
    x, y = x_coords[frame_idx], y_coords[frame_idx]
    
    # Fallback to center if tracking failed
    if np.isnan(x) or np.isnan(y): 
        x, y = w//2, h//2

    # Calculate top-left bounding box coordinates
    x1 = int(max(0, x - CROP_SIZE//2))
    y1 = int(max(0, y - CROP_SIZE//2))
    
    # Constrain to frame boundaries
    if x1 + CROP_SIZE > w: x1 = w - CROP_SIZE
    if y1 + CROP_SIZE > h: y1 = h - CROP_SIZE
    
    cropped = frame[y1:y1+CROP_SIZE, x1:x1+CROP_SIZE]
    out.write(cropped)
    
    offsets.append({'frame': frame_idx, 'x_offset': x1, 'y_offset': y1})
    frame_idx += 1

cap.release()
out.release()

pd.DataFrame(offsets).to_csv(os.path.join(os.path.dirname(STATIC_VIDEO), 'crop_offsets.csv'), index=False)
print(f"✅ Dynamic video saved to {out_path}\n✅ Offsets saved to crop_offsets.csv")

Cell 6: Stage 2 Setup (The Fine Model)
Initializes the final project on the dynamically cropped video. Extracts frames for your detailed labeling.

In [ ]:
DYNAMIC_VIDEO = '/mnt/Data/Projects/cloud_deployment/videos/Trial3/20260121_SERt3tp_dynamic_crop.mp4'

print("⚙️ Initializing Stage 2 Project...")
stage2_config = deeplabcut.create_new_project(
    'Stage2_Fine', 'Dev', [DYNAMIC_VIDEO], 
    working_directory=WORKING_DIR, copy_videos=False
)

with open(stage2_config, 'r') as f: cfg = ryaml.load(f)

cfg['bodyparts'] = ['Snout', 'DorsalFin', 'TailBase', 'TailTip']
cfg['skeleton'] = [['Snout', 'DorsalFin'], ['DorsalFin', 'TailBase'], ['TailBase', 'TailTip']]
cfg['default_net_type'] = 'mobilenet_v2_1.0'
cfg['numframes2pick'] = 150

with open(stage2_config, 'w') as f: ryaml.dump(cfg, f)

print(f"✅ Stage 2 initialized. Extracting frames...")
deeplabcut.extract_frames(stage2_config, mode='automatic', algo='kmeans', userfeedback=False)

Cell 7: Stage 2 Parse & Train
(Assumption: You have labeled the Stage 2 frames and have the CSV). Update LS_CSV_STAGE2. Notice the increased batch size and global scale since the video is much smaller and we are using MobileNet.

In [ ]:
LS_CSV_STAGE2 = '/path/to/your/stage2_labelstudio_export.csv'
S2_VIDEO_NAME = '20260121_SERt3tp_dynamic_crop'
S2_PROJECT_PATH = os.path.dirname(stage2_config)
bodyparts = ['Snout', 'DorsalFin', 'TailBase', 'TailTip']

# --- 1. Parser ---
df_ls = pd.read_csv(LS_CSV_STAGE2)
data_rows = []
for _, row in df_ls.iterrows():
    fname = os.path.basename(row['image']).split('-')[-1] if '-' in row['image'] else os.path.basename(row['image'])
    frame_data = {'image': f"labeled-data/{S2_VIDEO_NAME}/{fname}"}
    for bp in bodyparts: frame_data[f'{bp}_x'], frame_data[f'{bp}_y'] = np.nan, np.nan
    try:
        labels = json.loads(row['label' if 'label' in df_ls.columns else 'annotation'])
        if isinstance(labels, list) and 'result' in labels[0]: labels = labels[0]['result']
        for l in labels:
            v = l.get('value', {})
            bp = v.get('keypointlabels', [None])[0]
            if bp in bodyparts:
                frame_data[f'{bp}_x'] = (v['x'] * v.get('original_width', 800)) / 100.0
                frame_data[f'{bp}_y'] = (v['y'] * v.get('original_height', 800)) / 100.0
    except: pass
    data_rows.append(frame_data)

df_final = pd.DataFrame(data_rows).set_index('image')
df_final.columns = pd.MultiIndex.from_product([['Dev'], bodyparts, ['x', 'y']], names=['scorer', 'bodyparts', 'coords'])
frame_dest = os.path.join(S2_PROJECT_PATH, 'labeled-data', S2_VIDEO_NAME)
df_final.to_hdf(os.path.join(frame_dest, 'CollectedData_Dev.h5'), key='df_with_missing', mode='w')
df_final.to_csv(os.path.join(frame_dest, 'CollectedData_Dev.csv'))

# --- 2. Dataset Setup ---
deeplabcut.create_training_dataset(stage2_config, net_type='mobilenet_v2_1.0', augmenter_type='default')

# --- 3. VRAM Patches (Increased capability) ---
for p in glob.glob(os.path.join(S2_PROJECT_PATH, 'dlc-models', '*', '*', '*', 'pose_cfg.yaml'), recursive=True):
    with open(p, 'r') as f: text = f.read()
    text = re.sub(r'^global_scale:\s*.*$', 'global_scale: 0.8', text, flags=re.MULTILINE) # Higher res for tiny crop
    if 'train' in p: text = re.sub(r'^batch_size:\s*.*$', 'batch_size: 4', text, flags=re.MULTILINE) # Bumped
    with open(p, 'w') as f: f.write(text)

# --- 4. Train & Analyze ---
deeplabcut.train_network(stage2_config, maxiters=80000, displayiters=1000, allow_growth=True)
deeplabcut.analyze_videos(stage2_config, [DYNAMIC_VIDEO], save_as_csv=True, batchsize=4)
deeplabcut.create_labeled_video(stage2_config, [DYNAMIC_VIDEO], draw_skeleton=True)

Cell 8: Projection Mapping (The Final Printout)
This block takes the Stage 2 coordinates and adds the offsets from Cell 5, giving you the absolute X,Y tracking points as if they were predicted on the full video.

In [ ]:
# Load the offsets
offsets_df = pd.read_csv(os.path.join(os.path.dirname(STATIC_VIDEO), 'crop_offsets.csv'))

# Find the Stage 2 analysis CSV
s2_csv_files = glob.glob(os.path.join(os.path.dirname(STATIC_VIDEO), f'{S2_VIDEO_NAME}DLC*.csv'))
s2_track_csv = sorted(s2_csv_files)[-1]

# Load predictions
df_s2 = pd.read_csv(s2_track_csv, header=[0, 1, 2], index_col=0)
scorer = df_s2.columns[0][0]

# Add offsets back to the predictions
for bp in bodyparts:
    df_s2[(scorer, bp, 'x')] += offsets_df['x_offset'].values
    df_s2[(scorer, bp, 'y')] += offsets_df['y_offset'].values

# Save the absolute coordinate CSV
final_out = os.path.join(os.path.dirname(STATIC_VIDEO), 'FINAL_ABSOLUTE_TRACKING.csv')
df_s2.to_csv(final_out)
print(f"✅ Absolute tracking data restored and saved to: {final_out}")

Everything below here is dev pipeline:

In [2]:
import pandas as pd
import numpy as np
import cv2
import os

# --- Paths ---
CSV_PATH = '/mnt/Data/Projects/cloud_deployment/videos/20260318_SERedt/20260318SER_t7w1_redlinear_GX010260_MASKEDDLC_resnet50_Cloud_Deployment_SERMar28shuffle1_80000.csv'
VIDEO_PATH = '/mnt/Data/Projects/cloud_deployment/videos/20260318_SERedt/20260318SER_t7w1_redlinear_GX010260_MASKED.MP4'
OUT_VIDEO = '/mnt/Data/Projects/cloud_deployment/videos/20260318_SERedt/20260318SER_dynamic_crop.mp4'
OUT_OFFSETS = '/mnt/Data/Projects/cloud_deployment/videos/20260318_SERedt/crop_offsets.csv'

CROP_SIZE = 800
Y_CROP_OFFSET = 700  # <--- The missing translation 

print("📊 Calculating robust trajectory with Y-offset...")
df = pd.read_csv(CSV_PATH, header=[1, 2], index_col=0)
bodyparts = ['Snout', 'DorsalFin', 'TailBase', 'TailTip']

centroids_x = []
centroids_y = []

for idx, row in df.iterrows():
    xs, ys = [], []
    for bp in bodyparts:
        # Only consider points the model is at least 5% sure about
        if row[(bp, 'likelihood')] > 0.05:
            xs.append(row[(bp, 'x')])
            ys.append(row[(bp, 'y')] + Y_CROP_OFFSET) # <--- Applied here
    
    # Use median to ignore severe outliers
    if len(xs) >= 2:
        centroids_x.append(np.median(xs))
        centroids_y.append(np.median(ys))
    elif len(xs) == 1:
        centroids_x.append(xs[0])
        centroids_y.append(ys[0])
    else:
        centroids_x.append(np.nan)
        centroids_y.append(np.nan)

# Create a dataframe for the trajectory, interpolate missing frames, and smooth it
traj_df = pd.DataFrame({'x': centroids_x, 'y': centroids_y})
traj_df = traj_df.interpolate(method='linear', limit_direction='both')
traj_df['x'] = traj_df['x'].rolling(window=15, center=True, min_periods=1).mean()
traj_df['y'] = traj_df['y'].rolling(window=15, center=True, min_periods=1).mean()

print("✂️ Generating stabilized dynamic crop video...")
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUT_VIDEO, fourcc, fps, (CROP_SIZE, CROP_SIZE))

offsets = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret: break
    
    h, w, _ = frame.shape
    
    # Get smoothed centroid coordinates
    cx = traj_df['x'].iloc[frame_idx]
    cy = traj_df['y'].iloc[frame_idx]
    
    # Calculate top-left bounding box coordinates
    x1 = int(max(0, cx - CROP_SIZE // 2))
    y1 = int(max(0, cy - CROP_SIZE // 2))
    
    # Keep the box inside the frame boundaries
    if x1 + CROP_SIZE > w: x1 = w - CROP_SIZE
    if y1 + CROP_SIZE > h: y1 = h - CROP_SIZE
    
    cropped = frame[y1:y1+CROP_SIZE, x1:x1+CROP_SIZE]
    out.write(cropped)
    
    offsets.append({'frame': frame_idx, 'x_offset': x1, 'y_offset': y1})
    frame_idx += 1

cap.release()
out.release()

pd.DataFrame(offsets).to_csv(OUT_OFFSETS, index=False)
print(f"✅ Dynamic video saved to {OUT_VIDEO}")
print(f"✅ Offsets saved to {OUT_OFFSETS}")

📊 Calculating robust trajectory with Y-offset...
✂️ Generating stabilized dynamic crop video...
✅ Dynamic video saved to /mnt/Data/Projects/cloud_deployment/videos/20260318_SERedt/20260318SER_dynamic_crop.mp4
✅ Offsets saved to /mnt/Data/Projects/cloud_deployment/videos/20260318_SERedt/crop_offsets.csv
